# CodeAlpha Data Science Internship: Iris Flower Classification

**Author:** Data Science Intern  
**Project Repository:** `CodeAlpha_IrisFlowerClassification`  
**Task:** Task 1 - Iris Flower Classification  
**Goal:** Train and rigorously evaluate supervised machine learning models to classify iris flowers into three distinct species (*Iris setosa*, *Iris versicolor*, and *Iris virginica*) using sepal and petal measurements.

---

## 1. Problem Definition and Dataset Provenance
The Iris dataset is a landmark multivariate benchmark introduced by the British statistician and biologist Ronald A. Fisher in 1936 in *The Use of Multiple Measurements in Taxonomic Problems*. The physical measurements were gathered by Edgar Anderson on the Gaspé Peninsula in Quebec, Canada.

### Attributes and Units
- **Sepal Length**: Length of the flower's sepal in centimeters ($cm$)
- **Sepal Width**: Width of the flower's sepal in centimeters ($cm$)
- **Petal Length**: Length of the flower's petal in centimeters ($cm$)
- **Petal Width**: Width of the flower's petal in centimeters ($cm$)
- **Target Label**: Species variety (`setosa`, `versicolor`, or `virginica`)

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import joblib

# Set deterministic seeds and visual styles
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 5)

## 2. Dataset Loading and Initial Inspection
We load the dataset using `sklearn.datasets.load_iris(as_frame=True)` to prevent network-dependent download failures and ensure perfect reproducibility.

In [ ]:
iris_raw = load_iris(as_frame=True)
df = iris_raw.frame.copy()

# Standardize column naming for clean programmatic access
df.columns = [c.replace(" (cm)", "").replace(" ", "_") for c in df.columns]
df["species"] = df["target"].map(lambda i: iris_raw.target_names[i])

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Target species: {list(iris_raw.target_names)}")
display(df.head())

### Data Integrity and Verification Checks

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
duplicate_rows = df[df.duplicated(subset=feature_cols, keep=False)]
print(f"\nDuplicate feature rows detected: {len(duplicate_rows)} rows")
if len(duplicate_rows) > 0:
    display(duplicate_rows)

print("\nSpecies Distribution:")
print(df["species"].value_counts())

### Descriptive Statistics

In [ ]:
print("Descriptive statistics for feature measurements (cm):")
display(df[feature_cols].describe().round(3))

print("\nMean feature measurements grouped by species:")
display(df.groupby("species")[feature_cols].mean().round(3))

## 3. Exploratory Visualizations
Visualizing the relationships among features helps assess separability between classes before modeling.

In [ ]:
# Pairplot showing pair-wise relationships and KDE densities
g = sns.pairplot(df, hue="species", vars=feature_cols, diag_kind="kde", height=2.3)
g.figure.suptitle("Feature Pairplot across Iris Species", y=1.02, fontsize=14, weight="bold")
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(6, 5))
corr = df[feature_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f", square=True)
plt.title("Feature Pearson Correlation Heatmap", weight="bold", pad=12)
plt.show()

**Key EDA Insights:**
1. *Iris setosa* is linearly separable from the other two species based entirely on petal dimensions (`petal_length` and `petal_width`).
2. *Iris versicolor* and *Iris virginica* exhibit slight overlap in boundary regions, but can be distinguished with high confidence by combining petal width and sepal length.
3. Strong collinearity exists between `petal_length` and `petal_width` ($r \approx 0.96$).

## 4. Train-Test Split with Stratification
To avoid data leakage, preprocessing steps (such as feature standardisation) must only be learned from the training set. We reserve 20% of the data ($N = 30$) for unbiased final model testing using stratified sampling to guarantee equal class proportions.

In [ ]:
X = df[feature_cols]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples (40 per class)")
print(f"Held-out test set: {X_test.shape[0]} samples (10 per class)")

## 5. Model Training & Comparison via Scikit-Learn Pipelines
We evaluate four models:
1. **Baseline**: `DummyClassifier` with stratified random predictions to set our minimum acceptable performance floor.
2. **Logistic Regression**: Linear multiclass classifier with $L_2$ regularization and standardized features.
3. **Random Forest**: Ensemble non-linear decision forest (100 estimators, max depth 4).
4. **K-Nearest Neighbors**: Distance-based non-parametric classifier ($k = 5$).

In [ ]:
models = {
    "Baseline (Stratified Dummy)": Pipeline([
        ("clf", DummyClassifier(strategy="stratified", random_state=RANDOM_SEED))
    ]),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=300, random_state=RANDOM_SEED))
    ]),
    "Random Forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=100, max_depth=4, random_state=RANDOM_SEED))
    ]),
    "K-Nearest Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
results = {}

for name, pipe in models.items():
    # 5-fold CV score on training set only
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    
    # Fit on full training set and evaluate on test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    results[name] = {
        "CV Accuracy Mean": cv_scores.mean(),
        "CV Accuracy Std": cv_scores.std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision (Macro)": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Test Recall (Macro)": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Test F1 (Macro)": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

results_df = pd.DataFrame(results).T
display(results_df.round(4))

## 6. Confusion Matrix & Per-Class Diagnostics
Let us inspect the classification reports and confusion matrices on the held-out test set.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_models = ["Logistic Regression", "Random Forest", "K-Nearest Neighbors"]

for ax, name in zip(axes, plot_models):
    pipe = models[name]
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=iris_raw.target_names, yticklabels=iris_raw.target_names, ax=ax
    )
    ax.set_title(f"{name}\nAccuracy: {accuracy_score(y_test, y_pred):.1%}", weight="bold")
    ax.set_xlabel("Predicted Species")
    ax.set_ylabel("True Species")

plt.tight_layout()
plt.show()

## 7. Model Selection and Artifact Persistence
Based on both cross-validation stability and calibrated probabilistic predictions, Logistic Regression is chosen as our final production pipeline.

In [ ]:
champion_pipeline = models["Logistic Regression"]
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

artifact = {
    "pipeline": champion_pipeline,
    "feature_names": feature_cols,
    "target_names": list(iris_raw.target_names),
    "random_state": RANDOM_SEED
}

joblib.dump(artifact, models_dir / "iris_classifier.joblib")
print(f"Pipeline serialized successfully to {models_dir / 'iris_classifier.joblib'}")

## 8. Verification and Prediction Function
We verify that the saved pipeline can be reloaded and queried with sample measurements.

In [ ]:
loaded = joblib.load(models_dir / "iris_classifier.joblib")
sample_input = pd.DataFrame([{
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}])

predicted_idx = loaded["pipeline"].predict(sample_input)[0]
probabilities = loaded["pipeline"].predict_proba(sample_input)[0]

print(f"Predicted species: {loaded['target_names'][predicted_idx]}")
for name, prob in zip(loaded["target_names"], probabilities):
    print(f" - {name.capitalize()}: {prob*100:.2f}%")